# Hypothesis 1 — Adaptive MMNN tangent basis under oscillatory dynamics

This experiment now performs an actual DTB evolution for each
\(\omega\in\{\pi,4\pi,8\pi,16\pi\}\). Each frequency receives a fresh MMNN
with the same initialization seed and the same initial particle cloud, so the
frequency is the only changing condition.

At time \(t_k\), DTB recomputes the current neural tangent matrix
\(J_k=J_{\theta_k}(z)\), projects the game velocity,

\[
\alpha_k=\arg\min_\alpha\|J_k\alpha-b_\omega(X_k)\|_2^2,
\]

and advances both states with the same tangent increment,

\[
X_{k+1}=X_k+hJ_k\alpha_k,\qquad
\theta_{k+1}=\theta_k+h\alpha_k.
\]

The tangent inputs \(z\) are the fixed initial labels, while the accumulated
physical state \(X_k\) and parameters \(\theta_k\) evolve. Consequently the
basis changes through \(\theta_k\); there are no random resets or refits.
After the last update, the notebook performs a **new projection at
\((X_T,\theta_T)\)**. This is the reported final projection error.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Locate DTB_Ver3 in a local/PACE checkout. Clone the branch only in Colab.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((p for p in candidates if (p / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import (
    ExperimentConfig,
    OscillatoryNonpotentialGame,
    ResidualMMNN,
    evaluate_dtb_projection,
    run_experiment,
)
from DTB_Ver3.dtb import flat_parameters
from DTB_Ver3.models import count_parameters
from DTB_Ver3.utils import resolve_device, resolve_dtype, warmup_cuda, write_csv

output_dir = repo_root / 'DTB_Ver3' / 'results' / 'oscillatory_h1_adaptive_dtb'
output_dir.mkdir(parents=True, exist_ok=True)
print('repository:', repo_root)
print('output:', output_dir)


## 1. Experimental controls

The full trainable MMNN coordinate set is used and remains fixed. “Adaptive
basis” means that the Jacobian is recomputed at the updated parameter vector
\(	heta_k\) at every step. Every frequency uses the same particle seed and
MMNN seed for a matched comparison.

In [ ]:
SEED = 2026
MODEL_SEED = SEED + 100
KAPPA = 1.0
AMPLITUDE = 1.0
FREQUENCY_MULTIPLES = (1, 4, 8, 16)
OMEGAS = tuple(multiplier * np.pi for multiplier in FREQUENCY_MULTIPLES)

N_PARTICLES = 2000
STEP_SIZE = 0.002
FINAL_TIME = 0.2
SNAPSHOT_TIMES = (0.0, 0.05, 0.1, 0.2)
RK4_REFERENCE_STEP = 0.00025

MMNN_WIDTH = 12
MMNN_RANK = 12
MMNN_DEPTH = 3
ACTIVATION = 'tanh'
SVD_RTOL = 1e-8
JACOBIAN_CHUNK = 512

DEVICE_NAME = 'auto'
DTYPE_NAME = 'float64'
DEVICE = resolve_device(DEVICE_NAME)
DTYPE = resolve_dtype(DTYPE_NAME)
warmup_cuda(DEVICE, DTYPE)

print({
    'device': str(DEVICE),
    'dtype': str(DTYPE),
    'omega_over_pi': FREQUENCY_MULTIPLES,
    'particles': N_PARTICLES,
    'step_size': STEP_SIZE,
    'steps_per_frequency': int(round(FINAL_TIME / STEP_SIZE)),
})


## 2. Run one adaptive DTB experiment for each frequency

For each \(\omega\), the cell below:

1. constructs a fresh identity-initialized residual MMNN;
2. uses every trainable MMNN coordinate in a fixed tangent bundle;
3. projects the full game velocity \(b_\omega(X_k)\);
4. updates \(X_k\) and \(\theta_k\);
5. recomputes the tangent matrix at the next step;
6. compares the final particle cloud with a refined RK4 reference;
7. recomputes the full-velocity projection at \(T\);
8. separately projects \(q_\omega(X_T)\) using the same final adaptive basis.

The last diagnostic isolates representation of the oscillatory component, but
only the full-velocity projection drives the DTB evolution.

In [ ]:
runs = {}
models = {}
summary_rows = []
history_paths = []
initial_parameter_vectors = []

for multiplier, omega in zip(FREQUENCY_MULTIPLES, OMEGAS):
    print(f'\nAdaptive H1 run: omega={multiplier}pi', flush=True)

    # Reset to the same seed so every frequency starts from the same MMNN basis.
    torch.manual_seed(MODEL_SEED)
    model = ResidualMMNN(
        2,
        width=MMNN_WIDTH,
        rank=MMNN_RANK,
        depth=MMNN_DEPTH,
        activation=ACTIVATION,
        dtype=DTYPE,
        zero_init_output=True,
    ).to(DEVICE)
    parameter_count = count_parameters(model)

    game = OscillatoryNonpotentialGame(KAPPA, AMPLITUDE, float(omega))
    run_dir = output_dir / f'omega_{multiplier}pi'
    config = ExperimentConfig(
        dynamics='deterministic',
        run_reference=True,
        reference_integrator='rk4',
        reference_step_size=RK4_REFERENCE_STEP,
        particle_count=N_PARTICLES,
        initial_law='uniform',
        initial_low=-1.0,
        initial_high=1.0,
        step_size=STEP_SIZE,
        final_time=FINAL_TIME,
        snapshot_times=SNAPSHOT_TIMES,
        width=MMNN_WIDTH,
        rank=MMNN_RANK,
        depth=MMNN_DEPTH,
        activation=ACTIVATION,
        model_kind='residual_mmnn',
        zero_init_output=True,
        basis_size=parameter_count,
        subset_tangent_selection='fixed',
        tangent_input_mode='fixed_initial_labels',
        track_network_map=False,
        svd_rtol=SVD_RTOL,
        jacobian_chunk=JACOBIAN_CHUNK,
        seed=SEED,
        dtype=DTYPE_NAME,
        device=DEVICE_NAME,
        progress_reports=5,
        output_dir=run_dir,
        save_outputs=True,
    )
    result = run_experiment(game, config, model=model)
    runs[multiplier] = result
    models[multiplier] = model
    initial_parameter_vectors.append(result.initial_parameters.copy())

    # A separate final diagnostic for only q_omega, using theta_T and J_{theta_T}(z).
    _, structure = flat_parameters(model)
    theta_final = torch.as_tensor(result.final_parameters, dtype=DTYPE, device=DEVICE)
    labels = torch.as_tensor(result.initial_particles, dtype=DTYPE, device=DEVICE)
    final_particles = torch.as_tensor(
        result.dtb_final_particles, dtype=DTYPE, device=DEVICE
    )
    selected = torch.as_tensor(
        result.selected_indices, dtype=torch.long, device=DEVICE
    )
    final_oscillatory_target = game.oscillatory_velocity(final_particles)
    final_oscillatory_projection = evaluate_dtb_projection(
        theta_final,
        selected,
        labels,
        final_oscillatory_target,
        model,
        structure,
        chunk_size=JACOBIAN_CHUNK,
        svd_rtol=SVD_RTOL,
    )

    # Step diagnostics are evaluated at t_0,...,t_{K-1}; append the fresh t=T value.
    diagnostic_times = np.append(result.projection_times, result.times[-1])
    rms_history = np.append(result.projection_error, result.final_projection_error)
    relative_history = np.append(
        result.relative_projection_error,
        result.final_relative_projection_error,
    )
    alpha_history = np.append(result.alpha_norm, result.final_alpha_norm)
    history_path = write_csv(
        output_dir / f'omega_{multiplier}pi_projection_history.csv',
        ('time', 'rms_projection_error', 'relative_projection_error', 'alpha_norm'),
        zip(diagnostic_times, rms_history, relative_history, alpha_history),
    )
    history_paths.append(history_path)

    parameter_change = float(
        np.linalg.norm(result.final_parameters - result.initial_parameters)
    )
    summary_rows.append([
        multiplier,
        float(omega),
        parameter_count,
        float(result.projection_error[0]),
        float(result.relative_projection_error[0]),
        result.final_projection_error,
        result.final_relative_projection_error,
        float(final_oscillatory_projection.rms_residual.item()),
        float(final_oscillatory_projection.relative_residual.item()),
        result.final_alpha_norm,
        result.final_projection_rank,
        result.final_projection_condition,
        parameter_change,
        result.final_paired_rms,
    ])
    print({
        'omega_over_pi': multiplier,
        'initial_relative_projection_error': float(result.relative_projection_error[0]),
        'true_final_relative_projection_error': result.final_relative_projection_error,
        'final_oscillatory_relative_projection_error': float(
            final_oscillatory_projection.relative_residual.item()
        ),
        'final_DTB_RK4_RMS': result.final_paired_rms,
    })

summary_columns = (
    'omega_over_pi', 'omega', 'trainable_coordinates',
    'initial_full_rms_projection_error', 'initial_full_relative_projection_error',
    'final_full_rms_projection_error', 'final_full_relative_projection_error',
    'final_oscillatory_rms_projection_error',
    'final_oscillatory_relative_projection_error',
    'final_alpha_norm', 'final_retained_rank', 'final_condition_number',
    'parameter_change_norm', 'final_DTB_RK4_RMS',
)
summary_path = write_csv(
    output_dir / 'adaptive_h1_frequency_summary.csv',
    summary_columns,
    summary_rows,
)
maximum_initial_parameter_difference = max(
    np.max(np.abs(vector - initial_parameter_vectors[0]))
    for vector in initial_parameter_vectors
)
print('maximum initial parameter difference across frequencies:', maximum_initial_parameter_difference)
print('saved:', summary_path)


## 3. Projection history and true final-state errors

Every curve contains one extra endpoint at \(T\). That endpoint is evaluated
after the final parameter and particle update; it is not the projection used at
\(T-h\).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(FREQUENCY_MULTIPLES)))

for color, multiplier in zip(colors, FREQUENCY_MULTIPLES):
    result = runs[multiplier]
    diagnostic_times = np.append(result.projection_times, result.times[-1])
    relative_history = np.append(
        result.relative_projection_error,
        result.final_relative_projection_error,
    )
    alpha_history = np.append(result.alpha_norm, result.final_alpha_norm)
    axes[0, 0].semilogy(
        diagnostic_times, relative_history, color=color,
        label=fr'$\omega={multiplier}\pi$',
    )
    axes[0, 1].semilogy(diagnostic_times, alpha_history, color=color)

rows = np.asarray(summary_rows, dtype=float)
omega_over_pi = rows[:, 0]
axes[1, 0].semilogy(
    omega_over_pi, rows[:, 4], 'o--', color='0.5', label='initial full velocity'
)
axes[1, 0].semilogy(
    omega_over_pi, rows[:, 6], 'o-', color='tab:blue', label='final full velocity'
)
axes[1, 0].semilogy(
    omega_over_pi, rows[:, 8], 's-', color='tab:orange',
    label='final oscillatory component'
)
axes[1, 1].loglog(
    omega_over_pi, rows[:, 13], 'o-', color='tab:red'
)

axes[0, 0].set(title='Adaptive relative projection error', xlabel='time', ylabel='relative error')
axes[0, 0].legend()
axes[0, 1].set(title=r'Adaptive coefficient norm $\|\alpha_k\|_2$', xlabel='time', ylabel='norm')
axes[1, 0].set(title='Initial versus true final projection error', xlabel=r'$\omega/\pi$', ylabel='relative error')
axes[1, 0].legend()
axes[1, 1].set(title='Final DTB versus refined RK4', xlabel=r'$\omega/\pi$', ylabel='paired RMS')
for axis in axes.flat:
    axis.grid(True, alpha=0.3)

figure_path = output_dir / 'adaptive_h1_diagnostics.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', figure_path)


## 4. How to interpret Hypothesis 1

The primary statistic is `final_full_relative_projection_error`, which is

\[
\frac{\|J_{\theta_T}(z)\alpha_T-b_\omega(X_T)\|_2}
{\|b_\omega(X_T)\|_2}.
\]

`final_oscillatory_relative_projection_error` repeats the final projection
with \(q_\omega(X_T)\) as the target. If the full error is small but the
oscillatory-only error is large, the easily represented damping component is
masking a high-frequency representation problem. The RK4 error is a separate
trajectory-accuracy measurement; a small instantaneous projection error does
not by itself guarantee a small accumulated trajectory error.

In [ ]:
archive_path = Path(shutil.make_archive(
    str(output_dir),
    'zip',
    root_dir=output_dir,
))
print('result archive:', archive_path)
